# Upload Train/Val/Test Splits to Zenodo

Run this notebook in Google Colab to consolidate and upload the 3 parquet splits to Zenodo.

**Before running:**
1. Go to zenodo.org → your name → Applications → Personal access tokens → New token
2. Check `deposit:write` scope
3. Copy the token and paste it into the `ZENODO_TOKEN` variable below

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
import os
import pyarrow.parquet as pq

ZENODO_TOKEN = 'cJbfzlmJPRHNM4hPWaoRRAM0wUBzzmgOdZLXNBG02R2xs1ssG7G0fGVUVOqh'  # <-- paste your token here
BASE_PATH    = '/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features_encoded'
LOCAL_OUT    = '/content/consolidated'

## Token Test — Verify Zenodo access before proceeding

In [11]:
r = requests.get(
    'https://zenodo.org/api/deposit/depositions',
    params={'access_token': ZENODO_TOKEN}
)
print(f'Status: {r.status_code}')
if r.status_code == 200:
    print('Token is valid — proceed to Step 1')
else:
    print(f'Token error: {r.text[:300]}')
    print('Fix the token before continuing')

Status: 403
Token error: {"status": 403, "message": "Permission denied."}
Fix the token before continuing


## Step 1 — Consolidate parquet folders into single files

In [7]:
os.makedirs(LOCAL_OUT, exist_ok=True)

for name in ['train.parquet', 'val.parquet', 'test.parquet']:
    src = os.path.join(BASE_PATH, name)
    out = os.path.join(LOCAL_OUT, name)
    print(f'Consolidating {name}...')
    pq.write_table(pq.read_table(src), out)
    size_gb = os.path.getsize(out) / (1024**3)
    print(f'  Done: {size_gb:.2f} GB')

Consolidating train.parquet...
  Done: 1.56 GB
Consolidating val.parquet...
  Done: 0.23 GB
Consolidating test.parquet...
  Done: 0.20 GB


## Step 2 — Create a new Zenodo deposit

In [17]:
r = requests.post(
    'https://zenodo.org/api/deposit/depositions',
    params={'access_token': ZENODO_TOKEN},
    json={}
)
assert r.status_code == 201, f'Failed to create deposit: {r.status_code} {r.text}'

deposition_id = r.json()['id']
bucket_url    = r.json()['links']['bucket']

print(f'Deposition ID : {deposition_id}')
print(f'Bucket URL    : {bucket_url}')

Deposition ID : 20469826
Bucket URL    : https://zenodo.org/api/files/bf9d2945-cb67-4673-bf95-963c5039baef


## Step 3 — Upload the 3 parquet files

In [18]:
for name in ['train.parquet', 'val.parquet', 'test.parquet']:
    path = os.path.join(LOCAL_OUT, name)
    print(f'Uploading {name}...')
    with open(path, 'rb') as f:
        r = requests.put(
            f'{bucket_url}/{name}',
            params={'access_token': ZENODO_TOKEN},
            data=f
        )
    assert r.status_code in (200, 201), f'Upload failed: {r.status_code} {r.text}'
    print(f'  Done: {name} uploaded successfully')

Uploading train.parquet...
  Done: train.parquet uploaded successfully
Uploading val.parquet...
  Done: val.parquet uploaded successfully
Uploading test.parquet...
  Done: test.parquet uploaded successfully


## Step 4 — Add metadata

In [19]:
metadata = {
    'metadata': {
        'title': 'US Domestic Flight Arrival Delay — Train/Val/Test Splits (2018–2024)',
        'upload_type': 'dataset',
        'description': (
            'Train (2018–2022), validation (2023), and test (2024) parquet splits '
            'for the UCSD MDS DSC 288R Capstone project on predicting US domestic '
            'flight arrival delays. Features include BTS on-time performance data, '
            'NOAA weather observations, and engineered lag features. '
            'Target: ArrDel15 (binary, 1 if arrival delay >= 15 minutes).'
        ),
        'creators': [
            {'name': 'Group 5, DSC 288R Capstone, UCSD MDS'}
        ],
        'license': 'cc-zero',
        'keywords': [
            'flight delay', 'airline', 'BTS', 'classification',
            'machine learning', 'PySpark', 'parquet'
        ]
    }
}

r = requests.put(
    f'https://zenodo.org/api/deposit/depositions/{deposition_id}',
    params={'access_token': ZENODO_TOKEN},
    json=metadata
)
assert r.status_code == 200, f'Metadata update failed: {r.status_code} {r.text}'
print('Metadata added successfully')

Metadata added successfully


## Step 5 — Publish

In [20]:
r = requests.post(
    f'https://zenodo.org/api/deposit/depositions/{deposition_id}/actions/publish',
    params={'access_token': ZENODO_TOKEN}
)
assert r.status_code == 202, f'Publish failed: {r.status_code} {r.text}'

doi        = r.json().get('doi')
record_url = r.json().get('links', {}).get('record_html')

print(f'Published successfully!')
print(f'DOI        : {doi}')
print(f'Record URL : {record_url}')

Published successfully!
DOI        : 10.5281/zenodo.20469826
Record URL : https://zenodo.org/record/20469826


---
## Local Upload — `bts_with_weather_holiday.parquet`

Run these cells locally (not in Colab) to upload `bts_with_weather_holiday.parquet` as a new version of the existing Zenodo record.

**Before running:**
1. Go to zenodo.org → your name → Applications → Personal access tokens → New token
2. Check `deposit:write` scope
3. Paste the token into `ZENODO_TOKEN` below

In [ ]:
import requests
import os

ZENODO_TOKEN     = 'PASTE_YOUR_TOKEN_HERE'
EXISTING_RECORD  = 20469826
LOCAL_FILE       = '/Users/sripriya/ucsd/MDS/CapstoneProject/AirlineArrivalDelay/data/bts_with_weather_holiday.parquet'

# Verify token
r = requests.get(
    'https://zenodo.org/api/deposit/depositions',
    params={'access_token': ZENODO_TOKEN}
)
print(f'Token check: {r.status_code}')
if r.status_code == 200:
    print('Token valid — proceed')
else:
    print(f'Token error: {r.text[:200]}')

In [ ]:
# Create a new version of the existing record
r = requests.post(
    f'https://zenodo.org/api/deposit/depositions/{EXISTING_RECORD}/actions/newversion',
    params={'access_token': ZENODO_TOKEN}
)
assert r.status_code == 201, f'Failed to create new version: {r.status_code} {r.text[:300]}'

new_draft_url = r.json()['links']['latest_draft']
draft = requests.get(new_draft_url, params={'access_token': ZENODO_TOKEN}).json()
new_id     = draft['id']
bucket_url = draft['links']['bucket']

print(f'New draft ID : {new_id}')
print(f'Bucket URL   : {bucket_url}')

In [ ]:
# Upload bts_with_weather_holiday.parquet
filename = os.path.basename(LOCAL_FILE)
size_gb  = os.path.getsize(LOCAL_FILE) / (1024**3)
print(f'Uploading {filename} ({size_gb:.2f} GB)...')

with open(LOCAL_FILE, 'rb') as f:
    r = requests.put(
        f'{bucket_url}/{filename}',
        params={'access_token': ZENODO_TOKEN},
        data=f
    )
assert r.status_code in (200, 201), f'Upload failed: {r.status_code} {r.text[:300]}'
print(f'Uploaded successfully')

In [ ]:
# Publish the new version
r = requests.post(
    f'https://zenodo.org/api/deposit/depositions/{new_id}/actions/publish',
    params={'access_token': ZENODO_TOKEN}
)
assert r.status_code == 202, f'Publish failed: {r.status_code} {r.text[:300]}'

doi        = r.json().get('doi')
record_url = r.json().get('links', {}).get('record_html')
print(f'Published!')
print(f'DOI        : {doi}')
print(f'Record URL : {record_url}')